# Finding data of interest

The Epidata API includes numerous data streams -- medical claims data, cases and
deaths, mobility, and many others -- covering different geographic regions. This
can make it a challenge to find the data stream that you are most interested in.
This page will provide some advice on how to locate donate that may be useful to
you.

## Using the Delphi Epidata API documentation

The Delphi Epidata API documentation lists all the available data sources and
signals for
[COVID-19](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_signals.html)
and for [other
diseases](https://cmu-delphi.github.io/delphi-epidata/api/README.html#source-specific-parameters).
The site also includes a search tool if you have a keyword (e.g. "Taiwan") in
mind. Generally, any endpoint listed in the Delphi Epidata API has an associated
function in this client where its API endpoint name is prefixed with either
`pub_` or `pvt_`, e.g. `pub_covidcast` or `pvt_twitter`.

## Epidata data sources

The parameters available for each source data are documented in each linked
source-specific API page. The epidatpy client will also expect certain fields,
depending on the endpoint, though the Delphi Epidata API documentation will
contain more information about the accepted ranges of values for each field. 

A dynamically generated list of all available data sources can be obtained by
using the built-in `available_endpoints()`:

In [ ]:
# Hidden cell (set in the metadata for this cell)
import pandas as pd

# Set common options and context
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)
pd.set_option("display.width", 1000)

In [ ]:
from IPython.display import HTML

from epidatpy import available_endpoints

table = available_endpoints()
HTML(table.to_html(index=False))

## Covidcast source and signal metadata

The `CovidcastEpidata` class provides a way to access information about the data
in the `pub_covidcast` endpoint directly from within the client. The cell below
demonstrates how to access this metadata by using `source_df` property, which
returns a Pandas DataFrame of metadata describing all data streams publically
accessible from the COVIDcast endpoint of the Delphi Epidata API. This mirrors
the information found in the [COVIDcast signals
endpoint](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_signals.html).

In [ ]:
from epidatpy import CovidcastEpidata

epidata = CovidcastEpidata()
epidata.source_df

This DataFrame contains the following columns:

- `source` - API-internal source name.
- `name` - Human-readable source name.
- `description` - Description of the signal.
- `reference_signal` - Geographic level for which this signal is available, such as county, state, msa, hss, hrr, or nation. Most signals are available at multiple geographic levels and will hence be listed in multiple rows with their own metadata.
- `license` - The license.
- `dua` - Link to the Data Use Agreement.
- `signals` - List of signals available from this data source.

The `signal_df` DataFrame can also be used to obtain information about the signals
that are available - for example, what time range they are available for,
and when they have been updated.

In [ ]:
epidata.signal_df

This DataFrame contains one row each available signal, with the following columns:

- `source` - Data source name.
- `signal` - API-internal signal name.
- `name` - Human-readable signal name.
- `active` - Whether the signal is currently not updated or not. Signals may be inactive because the sources have become unavailable, other sources have replaced them, or additional work is required for us to continue updating them.
- `short_description` - Brief description of the signal.
- `description` - Full description of the signal.
- `geo_types` - Spatial resolution of the signal (e.g., `county`, `hrr`, `msa`, `dma`, `state`). More detail about all `geo_types` is given in the [geographic coding documentation](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_geography.html).
- `time_type` - Temporal resolution of the signal (e.g., day, week; see [date coding details](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_times.html)).
- `time_label` - The time label ("Date", "Week").
- `value_label` - The value label ("Value", "Percentage", "Visits", "Visits per 100,000 people").
- `format` - The value format ("per100k", "percent", "fraction", "count", "raw").
- `category` - The signal category ("early", "public", "late", "other").
- `high_values_are`- What the higher value of signal indicates ("good", "bad", "neutral").
- `is_smoothed` - Whether the signal is smoothed.
- `is_weighted` - Whether the signal is weighted.
- `is_cumulative` - Whether the signal is cumulative.
- `has_stderr` - Whether the signal has `stderr` statistic.
- `has_sample_size` - Whether the signal has `sample_size` statistic.
- `geo_types` - Geographical levels for which this signal is available.


## Example Queries

### Main Endpoint

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/covidcast_signals.html>

County geo_values are [FIPS codes](https://en.wikipedia.org/wiki/List_of_United_States_FIPS_codes_by_county) and are discussed in the API docs [here](https://cmu-delphi.github.io/delphi-epidata/api/covidcast_geography.html). The example below is for Orange County, California.


In [ ]:
from epidatpy import EpiDataContext, EpiRange
# Reuse existing context if available, but creating new one for standalone clarity
epidata = EpiDataContext()

epidata.pub_covidcast(
  data_source="fb-survey",
  signals="smoothed_accept_covid_vaccine",
  geo_type="county",
  time_type="day",
  time_values=EpiRange(20201221, 20201225),
  geo_values="06059"
).df()

### Other Covid Endpoints

#### COVID-19 Hospitalization: Facility Lookup

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/covid_hosp_facility_lookup.html>


In [ ]:
epidata.pub_covid_hosp_facility_lookup(city="southlake").df()

#### COVID-19 Hospitalization by Facility

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/covid_hosp_facility.html>


In [ ]:
epidata.pub_covid_hosp_facility(
  hospital_pks="100075",
  collection_weeks=EpiRange(20200101, 20200501)
).df()

#### COVID-19 Hospitalization by State

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/covid_hosp.html>


In [ ]:
epidata.pub_covid_hosp_state_timeseries(states="MA", dates="20200510").df()

### Flu Endpoints

#### FluSurv hospitalization data

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/flusurv.html>


In [ ]:
epidata.pub_flusurv(locations="ca", epiweeks=202001).df()

#### Fluview data

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/fluview.html>


In [ ]:
epidata.pub_fluview(regions="nat", epiweeks=EpiRange(201201, 202001)).df()

#### Delphi's ILINet forecasts

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/delphi.html>


In [ ]:
delphi_forecast = epidata.pub_delphi(system="ec", epiweek=201501)
delphi_forecast()['epidata']

#### FluView Clinical

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/fluview_clinical.html>


In [ ]:
epidata.pub_fluview_clinical(regions="nat", epiweeks=EpiRange(201601, 201701)).df()

#### FluView Metadata

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/fluview_meta.html>


In [ ]:
epidata.pub_fluview_meta().df()

#### Google Flu Trends

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/gft.html>


In [ ]:
epidata.pub_gft(locations="hhs1", epiweeks=EpiRange(201201, 202001)).df()

#### ECDC ILI

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/ecdc_ili.html>


In [ ]:
epidata.pub_ecdc_ili(regions="Armenia", epiweeks=201840).df()

#### KCDC ILI

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/kcdc_ili.html>


In [ ]:
epidata.pub_kcdc_ili(regions="ROK", epiweeks=200436).df()

#### NIDSS Flu

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nidss_flu.html>


In [ ]:
epidata.pub_nidss_flu(regions="taipei", epiweeks=EpiRange(200901, 201301)).df()

#### ILI Nearby Nowcast

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nowcast.html>


In [ ]:
epidata.pub_nowcast(locations="ca", epiweeks=EpiRange(202201, 202319)).df()

### Dengue Endpoints

#### Delphi's Dengue Nowcast

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/dengue_nowcast.html>


In [ ]:
epidata.pub_dengue_nowcast(locations="pr", epiweeks=EpiRange(201401, 202301)).df()

#### NIDSS Dengue

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/nidss_dengue.html>


In [ ]:
epidata.pub_nidss_dengue(locations="taipei", epiweeks=EpiRange(200301, 201301)).df()

#### PAHO Dengue

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/paho_dengue.html>


In [ ]:
epidata.pub_paho_dengue(regions="ca", epiweeks=EpiRange(200201, 202319)).df()

### Other Endpoints

#### Wikipedia Access

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/wiki.html>


In [ ]:
epidata.pub_wiki(
  language="en",
  articles="influenza",
  time_type="week",
  time_values=EpiRange(202001, 202319)
).df()

### Private methods

These require private access keys to use.

#### Google Health Trends

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/ght.html>


In [ ]:
# Requires valid API key
# epidata.pvt_ght(
#   auth="<YOUR_API_KEY>",
#   epiweeks=EpiRange(199301, 202304),
#   locations="ma",
#   query="how to get over the flu"
# ).df()

#### CDC

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/cdc.html>


In [ ]:
# epidata.pvt_cdc(auth="...", locations="ma", epiweeks=EpiRange(202003, 202304)).df()

#### Dengue Digital Surveillance Sensors

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/dengue_sensors.html>


In [ ]:
# epidata.pvt_dengue_sensors(
#   auth="...",
#   names="ght",
#   locations="ag",
#   epiweeks=EpiRange(201404, 202004)
# ).df()

#### NoroSTAT metadata

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/meta_norostat.html>


In [ ]:
# epidata.pvt_meta_norostat(auth="...").df()

#### NoroSTAT data

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/norostat.html>


In [ ]:
# epidata.pvt_norostat(auth="...", locations="1", epiweeks=201233).df()

#### Quidel Influenza testing

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/quidel.html>


In [ ]:
# epidata.pvt_quidel(auth="...", locations="hhs1", epiweeks=EpiRange(200301, 202105)).df()

#### Sensors

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/sensors.html>


In [ ]:
# epidata.pvt_sensors(
#   auth="...",
#   names="sar3",
#   locations="nat",
#   epiweeks=EpiRange(200301, 202105)
# ).df()

#### Twitter

API docs: <https://cmu-delphi.github.io/delphi-epidata/api/twitter.html>


In [ ]:
# epidata.pvt_twitter(
#   auth="...",
#   locations="nat",
#   time_type="week",
#   time_values=EpiRange(200301, 202105)
# ).df()